# CXR Disease Prediction: Patient-Safe BCE Baseline

Revised copy of CXR_Disease_Prediction. Original ResNet50 architecture and ImageNet V1 weights are retained.

- Patient-level multilabel stratification, with zero patient overlap assertions across train, validation, and official test lists.
- Unweighted BCE baseline (default); set LOSS_MODE to weighted_bce for a separate comparison with train-only negative/positive weights.
- Fourteen disease targets. No Finding images have an all-zero disease vector; this is a dataset annotation, not proof of health.
- Validation-derived per-disease F1 thresholds, frozen before test evaluation. The original already included threshold tuning; this copy fixes endpoint handling and saves thresholds.
- Per-disease ROC-AUC, average precision, positive/negative counts, precision, recall, and F1, plus a fixed-0.5 comparison.
- Missing/corrupt images fail visibly. Loss is averaged by sample count. Outputs are isolated by loss mode.

Run the cells in order with NIH Chest X-rays attached and GPU enabled. Training has not yet been run for this revision. Do not use the test set to select the loss mode; compare validation results first. Validation threshold-fit scores are optimistic; final performance comes from the untouched test set. Research use only.


In [ ]:
%pip install -q iterative-stratification


In [ ]:
# =============================
# Multi-Label Classification on NIH Chest X-ray Dataset
# =============================

import os
import random
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.models as models
import torch.optim as optim
import torch.nn.functional as F

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, precision_recall_curve, average_precision_score
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from torch.amp import GradScaler, autocast
from pathlib import Path
import json
from IPython.display import display
from torch.utils.tensorboard import SummaryWriter
import logging



## 1. Configuration


In [ ]:
# =============================
# 1. CONFIGURATION
# =============================
SEED = 42
BATCH_SIZE = 16
EPOCHS = 50
LOSS_MODE = "bce"  # Compare against "weighted_bce" in a separate run.
RUN_NAME = "resnet50_" + LOSS_MODE
OUTPUT_DIR = Path("/kaggle/working") / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Modify to match your dataset location:
ROOT_DIR = "/kaggle/input/data"  # The folder that contains images_001, ..., train_val_list.txt, etc.
# Locate the mounted dataset if its Kaggle slug differs from the original.
if not os.path.isfile(os.path.join(ROOT_DIR, "Data_Entry_2017.csv")):
    candidates = list(Path("/kaggle/input").rglob("Data_Entry_2017.csv"))
    if len(candidates) != 1:
        raise FileNotFoundError("Attach NIH Chest X-rays and set ROOT_DIR to its metadata folder.")
    ROOT_DIR = str(candidates[0].parent)
CSV_FILE = os.path.join(ROOT_DIR, "Data_Entry_2017.csv")
TRAIN_LIST_FILE = os.path.join(ROOT_DIR, "train_val_list.txt")
TEST_LIST_FILE  = os.path.join(ROOT_DIR, "test_list.txt")

# Define the diseases
DISEASES = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass", "Nodule",
    "Pneumonia", "Pneumothorax", "Consolidation", "Edema", "Emphysema",
    "Fibrosis", "Pleural_Thickening", "Hernia"
]
NUM_LABELS = len(DISEASES)

# Set random seeds for reproducibility
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Logging setup
logging.basicConfig(level=logging.INFO, filename=str(OUTPUT_DIR / 'training.log'), 
                    format='%(asctime)s %(levelname)s:%(message)s')
logger = logging.getLogger()



## 2. Load Csv + Train/Test Lists


In [ ]:
# =============================
# 2. LOAD CSV + TRAIN/TEST LISTS
# =============================
if not os.path.exists(CSV_FILE):
    raise FileNotFoundError(f"{CSV_FILE} not found.")

# Load the NIH label CSV (Data_Entry_2017.csv)
df = pd.read_csv(CSV_FILE)
df = df[["Image Index", "Finding Labels", "Patient ID"]]
assert df["Image Index"].is_unique, "Duplicate metadata rows"
assert not df.isna().any().any(), "Missing labels or patient IDs"
patient_map = df.set_index("Image Index")["Patient ID"].to_dict()
# No Finding is retained as an all-zero disease target, not a fifteenth disease.
known_labels = set(DISEASES) | {"No Finding"}
for labels in df["Finding Labels"]:
    parts = set(labels.split("|"))
    assert parts <= known_labels, f"Unknown labels: {parts - known_labels}"
    assert "No Finding" not in parts or len(parts) == 1, "Contradictory No Finding label"

# Create a dictionary: filename -> label string
labels_map = {}
for _, row in df.iterrows():
    filename = row["Image Index"]  # e.g. "00000003_000.png"
    label_str = row["Finding Labels"]  # e.g. "Cardiomegaly|Effusion"
    labels_map[filename] = label_str

# Load the train/val and test split files
def load_split_list(list_path):
    """Returns a list of image filenames from train_val_list.txt or test_list.txt."""
    if not os.path.exists(list_path):
        raise FileNotFoundError(f"{list_path} does not exist.")
    with open(list_path, 'r') as f:
        return [line.strip() for line in f if line.strip()]

train_val_filenames = load_split_list(TRAIN_LIST_FILE)
test_filenames      = load_split_list(TEST_LIST_FILE)

logger.info(f"Train/Val images: {len(train_val_filenames)}, Test images: {len(test_filenames)}")
print(f"Loaded {len(train_val_filenames)} training/validation images and {len(test_filenames)} test images.")



## 3. Dataset Class


In [ ]:
# =============================
# 3. DATASET CLASS
# =============================
class NIHChestXRayDataset(Dataset):
    def __init__(self, filenames, root_dir, labels_map, transform=None):
        """
        filenames: list of image filenames, e.g. ["00000003_000.png", ...]
        root_dir: folder containing images_00X/images/ and CSV
        labels_map: dict mapping filename -> label string
        transform: torchvision (or Albumentations) transforms
        """
        self.filenames = filenames
        self.root_dir = root_dir
        self.labels_map = labels_map
        self.transform = transform
        
        # Build a dictionary that maps "00000003_000.png" -> "full/path/to/image"
        self.path_map = {}
        # Look for subfolders: images_001, images_002, ... that each contain an 'images' folder
        subfolders = [d for d in os.listdir(root_dir) if d.startswith("images_")]
        
        for sf in subfolders:
            images_subdir = os.path.join(root_dir, sf, "images")
            if not os.path.isdir(images_subdir):
                continue
            # For each file in images_00X/images
            for fn in os.listdir(images_subdir):
                full_path = os.path.join(images_subdir, fn)
                self.path_map[fn] = full_path
        
        missing = [fn for fn in self.filenames if fn not in self.path_map]
        if missing:
            raise FileNotFoundError(f"{len(missing)} missing images; examples: {missing[:5]}")
    
    def __len__(self):
        return len(self.filenames)
    
    def __getitem__(self, idx):
        filename = self.filenames[idx]
        img_path = self.path_map[filename]
        
        # Fail visibly instead of training on a fabricated black image.
        with Image.open(img_path) as source:
            image = source.convert("RGB")
        
        # Apply transforms if any
        if self.transform:
            image = self.transform(image)
        
        # Convert "Finding Labels" => multi-label tensor
        label_str = self.labels_map.get(filename, "")  # e.g. "Effusion|Cardiomegaly"
        label_split = label_str.split("|") if label_str else []
        
        label_tensor = torch.zeros(NUM_LABELS, dtype=torch.float)
        for disease in label_split:
            if disease in DISEASES:
                idx_ = DISEASES.index(disease)
                label_tensor[idx_] = 1.0
            elif disease != "No Finding":
                logger.warning(f"Unexpected disease label '{disease}' in file {filename}.")
        
        return image, label_tensor



## 4. Data Preprocessing & Dataloaders


In [ ]:
# =============================
# 4. DATA PREPROCESSING & DATALOADERS
# =============================

# Define normalization based on ImageNet statistics
normalize = T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])

# Data augmentation for training
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
    T.ToTensor(),  # Convert PIL Image to Tensor before RandomErasing
    normalize,
    T.RandomErasing(p=0.1),  # Now correctly applied on Tensor
])

# No augmentation for validation and testing
test_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    normalize,
])

# Stratify patient-level label unions, then map patients back to images.
# The official test list is never used for model or threshold selection.
assert len(train_val_filenames) == len(set(train_val_filenames))
assert len(test_filenames) == len(set(test_filenames))
assert set(train_val_filenames).isdisjoint(test_filenames)
assert set(train_val_filenames + test_filenames) <= set(labels_map)
train_val_patients = {patient_map[fn] for fn in train_val_filenames}
test_patients = {patient_map[fn] for fn in test_filenames}
assert train_val_patients.isdisjoint(test_patients), "Official lists overlap by patient; review source splits."
mlb = MultiLabelBinarizer(classes=DISEASES)
mlb.fit([DISEASES])
def disease_labels(fn):
    return [x for x in labels_map[fn].split("|") if x != "No Finding"]
patient_labels = {}
for fn in train_val_filenames:
    patient_labels.setdefault(patient_map[fn], set()).update(disease_labels(fn))
patients = sorted(patient_labels)
patient_y = mlb.transform([sorted(patient_labels[p]) for p in patients])
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(msss.split(np.zeros((len(patients), 1)), patient_y))
train_patients = {patients[i] for i in train_idx}
val_patients = {patients[i] for i in val_idx}
assert train_patients.isdisjoint(val_patients)
assert train_patients.isdisjoint(test_patients)
assert val_patients.isdisjoint(test_patients)
train_filenames = [fn for fn in train_val_filenames if patient_map[fn] in train_patients]
val_filenames = [fn for fn in train_val_filenames if patient_map[fn] in val_patients]
for split, files in [("train", train_filenames), ("validation", val_filenames), ("test", test_filenames)]:
    pd.DataFrame({"filename": files, "patient_id": [patient_map[f] for f in files]}).to_csv(OUTPUT_DIR / f"{split}_split.csv", index=False)
    print(split, "images:", len(files), "patients:", len({patient_map[f] for f in files}))
prevalence = pd.DataFrame({name: mlb.transform([disease_labels(f) for f in files]).mean(axis=0)
    for name, files in [("train", train_filenames), ("validation", val_filenames), ("test", test_filenames)]}, index=DISEASES)
display(prevalence)
prevalence.to_csv(OUTPUT_DIR / "split_prevalence.csv")

# Initialize Datasets
train_dataset = NIHChestXRayDataset(train_filenames, ROOT_DIR, labels_map, transform=train_transform)
val_dataset   = NIHChestXRayDataset(val_filenames, ROOT_DIR, labels_map, transform=test_transform)
test_dataset  = NIHChestXRayDataset(test_filenames, ROOT_DIR, labels_map, transform=test_transform)

# Train-only positive weights; do not scale every loss by inverse counts.
binary_train_labels = mlb.transform([disease_labels(fn) for fn in train_dataset.filenames])
class_counts = binary_train_labels.sum(axis=0)
assert np.all(class_counts > 0), "A disease has no training positives"
assert np.all(class_counts < len(train_dataset)), "A disease has no training negatives"
pos_weight = torch.tensor((len(train_dataset) - class_counts) / class_counts, dtype=torch.float32, device=device)
if LOSS_MODE == "bce":
    criterion = nn.BCEWithLogitsLoss()
elif LOSS_MODE == "weighted_bce":
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
else:
    raise ValueError("LOSS_MODE must be bce or weighted_bce")
print("Loss:", LOSS_MODE, "Train-only pos_weight:", pos_weight)
pd.DataFrame({"disease": DISEASES, "train_positives": class_counts, "pos_weight": pos_weight.cpu().numpy()}).to_csv(OUTPUT_DIR / "class_weights.csv", index=False)

# Initialize DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

logger.info(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}, Test samples: {len(test_dataset)}")
print(f"DataLoaders initialized with Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)} samples.")



## 5. Model, Loss, & Training Loop


In [ ]:
# =============================
# 5. MODEL, LOSS, & TRAINING LOOP
# =============================
# Initialize the model (ResNet50) with pre-trained weights
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, NUM_LABELS)
model = model.to(device)
print("Initialized ResNet50 model with modified final layer.")

# Define optimizer (AdamW is often better for fine-tuning)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
print("Initialized AdamW optimizer.")

# Define learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)
print("Initialized ReduceLROnPlateau scheduler.")

# Initialize GradScaler for mixed precision training
scaler = GradScaler("cuda", enabled=device.type == "cuda")
print("Initialized GradScaler for mixed precision training.")

# Initialize TensorBoard writer
writer = SummaryWriter(log_dir=str(OUTPUT_DIR / "tensorboard"))
print("Initialized TensorBoard SummaryWriter.")

# Define training function
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    running_loss = 0.0
    for batch_idx, (images, labels) in enumerate(loader, 1):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        
        with autocast(device_type=device.type, enabled=device.type == "cuda"):
            outputs = model(images)
            loss = criterion(outputs, labels)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite loss")
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        
        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(f"Batch {batch_idx}/{len(loader)}: Loss={loss.item():.6f}")
    
    average_loss = running_loss / len(loader.dataset)
    return average_loss

# Define evaluation function with metrics
@torch.no_grad()
def evaluate_metrics(model, loader, criterion, threshold=0.5):
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_outputs = []
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        
        all_labels.append(labels.cpu().numpy())
        all_outputs.append(torch.sigmoid(outputs).cpu().numpy())
    
    all_labels = np.vstack(all_labels)
    all_outputs = np.vstack(all_outputs)
    
    # Calculate AUC for each class
    aucs = []
    for i in range(NUM_LABELS):
        if np.sum(all_labels[:, i]) == 0:
            continue  # Skip classes with no samples
        try:
            auc = roc_auc_score(all_labels[:, i], all_outputs[:, i])
            aucs.append(auc)
        except ValueError:
            logger.warning(f"ROC AUC score not defined for class index {i}.")
            if verbose:
                print(f"ROC AUC score not defined for class '{DISEASES[i]}'. Skipping.")
    
    mean_auc = np.mean(aucs) if aucs else 0
    
    # Binarize predictions with a threshold (e.g., 0.5)
    preds = (all_outputs >= threshold).astype(int)
    
    # Calculate F1, Precision, Recall
    f1 = f1_score(all_labels, preds, average='macro', zero_division=0)
    precision = precision_score(all_labels, preds, average='macro', zero_division=0)
    recall = recall_score(all_labels, preds, average='macro', zero_division=0)
    
    avg_loss = total_loss / len(loader.dataset)
    
    if verbose:
        print(f"Validation Metrics - Loss: {avg_loss:.6f}, AUC: {mean_auc:.6f}, F1: {f1:.6f}, Precision: {precision:.6f}, Recall: {recall:.6f}")
    
    return avg_loss, mean_auc, f1, precision, recall

# Function to find best thresholds based on validation set
def find_best_thresholds(model, loader, criterion):
    model.eval()
    all_labels = []
    all_outputs = []
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_labels.append(labels.numpy())
            all_outputs.append(probs)
    
    all_labels = np.vstack(all_labels)
    all_outputs = np.vstack(all_outputs)
    
    best_thresholds = []
    for i in range(NUM_LABELS):
        if np.sum(all_labels[:, i]) == 0:
            best_thresholds.append(0.5)  # Default threshold
            continue
        precision_vals, recall_vals, thresholds = precision_recall_curve(all_labels[:, i], all_outputs[:, i])
        f1_scores = 2 * (precision_vals[:-1] * recall_vals[:-1]) / (precision_vals[:-1] + recall_vals[:-1] + 1e-8)
        best_idx = np.argmax(f1_scores)
        best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
        best_thresholds.append(best_threshold)
        logger.info(f"Best threshold for class '{DISEASES[i]}': {best_threshold:.6f}")
        print(f"Best threshold for class '{DISEASES[i]}': {best_threshold:.6f}")
    
    return best_thresholds

# Function to apply best thresholds and calculate metrics
@torch.no_grad()
def evaluate_with_best_thresholds(model, loader, criterion, best_thresholds):
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_outputs = []
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        
        all_labels.append(labels.cpu().numpy())
        all_outputs.append(torch.sigmoid(outputs).cpu().numpy())
    
    all_labels = np.vstack(all_labels)
    all_outputs = np.vstack(all_outputs)
    
    preds = np.zeros_like(all_outputs)
    for i in range(NUM_LABELS):
        preds[:, i] = (all_outputs[:, i] >= best_thresholds[i]).astype(int)
    
    # Calculate metrics
    # Calculate AUC for each class
    aucs = []
    for i in range(NUM_LABELS):
        if np.sum(all_labels[:, i]) == 0:
            continue  # Skip classes with no samples
        try:
            auc = roc_auc_score(all_labels[:, i], all_outputs[:, i])
            aucs.append(auc)
        except ValueError:
            logger.warning(f"ROC AUC score not defined for class index {i}.")
    
    mean_auc = np.mean(aucs) if aucs else 0
    
    # Calculate F1, Precision, Recall
    f1 = f1_score(all_labels, preds, average='macro', zero_division=0)
    precision = precision_score(all_labels, preds, average='macro', zero_division=0)
    recall = recall_score(all_labels, preds, average='macro', zero_division=0)
    
    # Calculate class-wise metrics
    class_f1 = f1_score(all_labels, preds, average=None, zero_division=0)
    class_precision = precision_score(all_labels, preds, average=None, zero_division=0)
    class_recall = recall_score(all_labels, preds, average=None, zero_division=0)
    
    avg_loss = total_loss / len(loader.dataset)
    
    print(f"Test Metrics with Optimized Thresholds - Loss: {avg_loss:.6f}, AUC: {mean_auc:.6f}, F1: {f1:.6f}, Precision: {precision:.6f}, Recall: {recall:.6f}")
    logger.info(f"Test Set with Optimized Thresholds: Loss={avg_loss:.6f}, AUC={mean_auc:.6f}, F1={f1:.6f}, Precision={precision:.6f}, Recall={recall:.6f}")
    
    rows = []
    for i, disease in enumerate(DISEASES):
        y, p = all_labels[:, i], all_outputs[:, i]
        both = len(np.unique(y)) == 2
        rows.append({"disease": disease, "positives": int(y.sum()), "negatives": int((1-y).sum()),
                     "threshold": float(best_thresholds[i]),
                     "roc_auc": roc_auc_score(y, p) if both else np.nan,
                     "average_precision": average_precision_score(y, p) if y.sum() > 0 else np.nan,
                     "precision": class_precision[i], "recall": class_recall[i], "f1": class_f1[i]})
    report = pd.DataFrame(rows)
    display(report)
    report.to_csv(OUTPUT_DIR / "test_per_disease.csv", index=False)
    np.savez_compressed(OUTPUT_DIR / "test_predictions.npz", labels=all_labels, probabilities=all_outputs, diseases=np.array(DISEASES))
    default_preds = (all_outputs >= 0.5).astype(int)
    print("Test macro F1 at 0.5:", f1_score(all_labels, default_preds, average="macro", zero_division=0))
    print("Test macro average precision:", report["average_precision"].mean())
    
    # Log class-wise metrics
    print("\nClass-wise Metrics:")
    for i, disease in enumerate(DISEASES):
        if np.sum(all_labels[:, i]) == 0:
            continue
        print(f"{disease}: F1={class_f1[i]:.6f}, Precision={class_precision[i]:.6f}, Recall={class_recall[i]:.6f}")
        logger.info(f"{disease}: F1={class_f1[i]:.6f}, Precision={class_precision[i]:.6f}, Recall={class_recall[i]:.6f}")
    
    return avg_loss, mean_auc, f1, precision, recall

# Function to visualize prediction probabilities
def plot_prediction_distribution(all_outputs, all_labels, disease_index, disease_name):
    plt.figure(figsize=(8,6))
    plt.hist(all_outputs[:, disease_index][all_labels[:, disease_index]==1], bins=50, alpha=0.6, label='Positive')
    plt.hist(all_outputs[:, disease_index][all_labels[:, disease_index]==0], bins=50, alpha=0.6, label='Negative')
    plt.title(f'Prediction Probabilities for {disease_name}')
    plt.xlabel('Probability')
    plt.ylabel('Frequency')
    plt.legend()
    plt.show()

# Function to visualize some predictions
def visualize_predictions(dataset, model, device, num_samples=5, thresholds=None):
    """
    Visualize a few predictions alongside true labels.
    """
    model.eval()
    for i in range(num_samples):
        img, label = dataset[i]
        img_input = img.unsqueeze(0).to(device)
        with torch.no_grad():
            output = model(img_input)
            prob = torch.sigmoid(output).cpu().numpy()[0]
            if thresholds is not None:
                pred = (prob >= thresholds).astype(int)
            else:
                pred = (prob >= 0.5).astype(int)
            pred_labels = [DISEASES[j] for j, val in enumerate(pred) if val == 1]
            true_labels = [DISEASES[j] for j, val in enumerate(label) if val == 1]
        
        # Convert tensor to numpy array and unnormalize for visualization
        img_np = img.permute(1, 2, 0).numpy()
        img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_np = np.clip(img_np, 0, 1)
        
        plt.figure(figsize=(6,6))
        plt.imshow(img_np)
        plt.axis('off')
        plt.title(f"True: {true_labels}\nPredicted: {pred_labels}")
        plt.show()



## 6. Training Loop With Bce Loss


In [ ]:
# =============================
# 6. TRAINING LOOP WITH BCE LOSS
# =============================
# Compare each loss mode in its own output folder.
patience = 7  # Increased patience
best_loss = float('inf')
counter = 0
early_stop = False

logger.info("Starting training...")
print("Starting training...")

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    logger.info(f"Epoch {epoch+1}/{EPOCHS}")
    
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    print(f"Training Loss: {train_loss:.6f}")
    logger.info(f"Training Loss: {train_loss:.6f}")
    
    val_loss, val_auc, val_f1, val_precision, val_recall = evaluate_metrics(model, val_loader, criterion, threshold=0.5)
    
    scheduler.step(val_loss)
    
    # Log metrics to TensorBoard
    writer.add_scalar('Loss/Train', train_loss, epoch)
    writer.add_scalar('Loss/Val', val_loss, epoch)
    writer.add_scalar('AUC/Val', val_auc, epoch)
    writer.add_scalar('F1/Val', val_f1, epoch)
    writer.add_scalar('Precision/Val', val_precision, epoch)
    writer.add_scalar('Recall/Val', val_recall, epoch)
    
    # Check for improvement
    if val_loss < best_loss:
        best_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), str(OUTPUT_DIR / 'best_model.pth'))
        logger.info("Model improved. Saving best model.")
        print("Model improved. Saving best model.")
    else:
        counter += 1
        logger.info(f"No improvement. Early stopping counter: {counter}/{patience}")
        print(f"No improvement. Early stopping counter: {counter}/{patience}")
        if counter >= patience:
            logger.info("Early stopping triggered.")
            print("Early stopping triggered.")
            early_stop = True
            break

writer.close()

# Load the best model
model.load_state_dict(torch.load(str(OUTPUT_DIR / 'best_model.pth'), map_location=device, weights_only=True))
logger.info("Loaded the best model.")
print("Loaded the best model.")



## 7. Find Best Thresholds


In [ ]:
# =============================
# 7. FIND BEST THRESHOLDS
# =============================
logger.info("Finding best thresholds based on validation set...")
print("\nFinding best thresholds based on validation set...")
best_thresholds = find_best_thresholds(model, val_loader, criterion)
with open(OUTPUT_DIR / "thresholds.json", "w") as f:
    json.dump(dict(zip(DISEASES, map(float, best_thresholds))), f, indent=2)
with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump({"seed": SEED, "batch_size": BATCH_SIZE, "epochs": EPOCHS,
               "loss": LOSS_MODE, "labels": DISEASES, "torch": torch.__version__,
               "threshold_objective": "validation per-disease F1", "no_finding": "all-zero disease vector"}, f, indent=2)



## 8. Testing The Model With Best Thresholds


In [ ]:
# =============================
# 8. TESTING THE MODEL WITH BEST THRESHOLDS
# =============================
print("\nEvaluating model on test set with optimized thresholds...")
test_loss, test_auc, test_f1, test_precision, test_recall = evaluate_with_best_thresholds(
    model, test_loader, criterion, best_thresholds
)



## 9. Optional: Visualize Predictions


In [ ]:
# =============================
# 9. OPTIONAL: VISUALIZE PREDICTIONS
# =============================
print("\nVisualizing sample predictions from the test set...")
visualize_predictions(test_dataset, model, device, num_samples=5, thresholds=best_thresholds)



## 10. Additional Diagnostics


In [ ]:
# =============================
# 10. ADDITIONAL DIAGNOSTICS
# =============================

# Plot prediction distributions for each class
def plot_prediction_distributions(model, loader, device):
    model.eval()
    all_labels = []
    all_outputs = []
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_labels.append(labels.numpy())
            all_outputs.append(probs)
    
    all_labels = np.vstack(all_labels)
    all_outputs = np.vstack(all_outputs)
    
    for i, disease in enumerate(DISEASES):
        plt.figure(figsize=(8,6))
        plt.hist(all_outputs[all_labels[:, i]==1, i], bins=50, alpha=0.6, label='Positive')
        plt.hist(all_outputs[all_labels[:, i]==0, i], bins=50, alpha=0.6, label='Negative')
        plt.title(f'Prediction Probabilities for {disease}')
        plt.xlabel('Probability')
        plt.ylabel('Frequency')
        plt.legend()
        plt.show()

# Uncomment the following line to plot prediction distributions
# plot_prediction_distributions(model, val_loader, device)